# 🔡 Traductor Neuronal Shiwilu ↔ Español a Nivel de Caracteres
**Objetivo:** Entrenar un modelo de traducción automática neuronal (NMT) a nivel de caracteres con corpus paralelo Shiwilu–Español.

Este enfoque es útil para lenguas de escasos recursos donde no existen tokenizadores adecuados o vocabularios estandarizados.

🧪 Tecnologías: `Transformers`, `datasets`, entrenamiento desde cero (sin preentrenamiento).


## 1️⃣ Preparación del entorno

In [ ]:
!pip install -q transformers datasets sacrebleu

## 2️⃣ Carga del corpus y conversión a caracteres

In [ ]:
from datasets import Dataset

# Leer archivos
with open("shiwilu.txt", encoding='utf-8') as f:
    shiwilu = [list(line.strip()) for line in f.readlines()]

with open("espanol.txt", encoding='utf-8') as f:
    espanol = [list(line.strip()) for line in f.readlines()]

examples = [{"shw": ''.join(s), "es": ''.join(e)} for s, e in zip(shiwilu, espanol)]
data = Dataset.from_list(examples).train_test_split(test_size=0.1)
data

## 3️⃣ Tokenización a nivel de caracteres

In [ ]:
from transformers import PreTrainedTokenizerFast

chars = sorted(set(''.join([''.join(x) for x in shiwilu + espanol])))
vocab = {c: i+4 for i, c in enumerate(chars)}  # Reservar ids: 0-pad, 1-unk, 2-bos, 3-eos
vocab["[PAD]"] = 0
vocab["[UNK]"] = 1
vocab["[BOS]"] = 2
vocab["[EOS]"] = 3

tokenizer = PreTrainedTokenizerFast(tokenizer_object=None)
tokenizer.add_tokens(list(vocab.keys()))
tokenizer.pad_token = "[PAD]"
tokenizer.unk_token = "[UNK]"
tokenizer.bos_token = "[BOS]"
tokenizer.eos_token = "[EOS]"


## 4️⃣ Preparación para el entrenamiento

In [ ]:
from transformers import EncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments

# Modelo encoder-decoder base (desde cero)
model = EncoderDecoderModel.from_encoder_decoder_pretrained("bert-base-uncased", "bert-base-uncased")
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
# model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.encoder.resize_token_embeddings(len(tokenizer))
model.decoder.resize_token_embeddings(len(tokenizer))

def preprocess(example):
    inputs = tokenizer(example['shw'], truncation=True, padding='max_length', max_length=128, return_tensors='pt')
    targets = tokenizer(example['es'], truncation=True, padding='max_length', max_length=128, return_tensors='pt')
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_data = data.map(preprocess, batched=True, remove_columns=["shw", "es"])


## 5️⃣ Entrenamiento del modelo

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./char_nmt_shiwilu",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"]
)
# trainer.train()  # Descomenta para entrenar

## 6️⃣ Traducción de prueba

In [ ]:
from transformers import pipeline

translator = pipeline("translation", model=model, tokenizer=tokenizer)
print(translator("ashintu niwan"))

## 📏 Evaluación del Modelo con BLEU y Ejemplos de Traducción

In [ ]:
from datasets import load_metric
import numpy as np

# ⚠️ BLEU ya no está en load_metric, se puede usar sacrebleu directamente
from sacrebleu import corpus_bleu

# Traducciones del conjunto de prueba
refs = []
hyps = []

for example in data['test']:
    source = example['shw']
    reference = example['es']
    generated = translator(source)[0]['translation_text']
    refs.append([reference])
    hyps.append(generated)

bleu = corpus_bleu(hyps, list(zip(*refs)))
print(f"BLEU score: {bleu.score:.2f}")

### 🔍 Ejemplos de traducción generados por el modelo

In [ ]:
# Mostrar algunos ejemplos
for i in range(5):
    entrada = data['test'][i]['shw']
    referencia = data['test'][i]['es']
    traduccion = translator(entrada)[0]['translation_text']
    print(f"🔸 Entrada:     {entrada}")
    print(f"🎯 Referencia: {referencia}")
    print(f"🤖 Traducción: {traduccion}\n")